# Introduction

This notebook shows the effect of Newton-Raphson method for state estimation. We create a small network with two power sensors to show how Newton-Raphson method can split the uncertainties of active and reactive power in the sensors.

# Example network

The network used in the notebook is shown below.

```
source_1-------- node_0 -------------------------line_2-------- node_3 -----------sym_load_4
                  |                     |                                   |
           voltage_sensor_5         power_sensor_6                       power_sensor_7
           u_measured = 10.5e3      p_measured = 1e6                     p_measured = -1e6
           u_sigma = 100.0          q_measured = 1e6                     q_measured = -1e6
                                    p_sigma = 1e4                        p_sigma = 1e9
                                    q_sigma = 1e9                        q_sigma = 1e4
```

The network has two power sensors which have completely opposity measured value. However, the accuracy (error range in sigma) for both sensors are opposite for active and reactive power. So in theory, we should trust the `power_sensor_6` for `p_measured` and `power_sensor_7` for `q_measured`.

# Imports

In [1]:
import pandas as pd
import numpy as np

from power_grid_model import PowerGridModel, LoadGenType, MeasuredTerminalType, initialize_array, CalculationType, CalculationMethod
from power_grid_model.validation import assert_valid_input_data


# Construct network input

In [2]:
# node
node = initialize_array("input", "node", 2)
node["id"] = np.array([0, 3])
node["u_rated"] = [10.5e3, 10.5e3]

# line
line = initialize_array("input", "line", 1)
line["id"] = [2]
line["from_node"] = [0]
line["to_node"] = [3]
line["from_status"] = [1]
line["to_status"] = [1]
line["r1"] = [0.001]
line["x1"] = [0.02]
line["c1"] = [0.0]
line["tan1"] = [0.0]
line["i_n"] = [1000.0]

# load
sym_load = initialize_array("input", "sym_load", 1)
sym_load["id"] = [4]
sym_load["node"] = [3]
sym_load["status"] = [1]
sym_load["type"] = [LoadGenType.const_power]
sym_load["p_specified"] = [1e6]
sym_load["q_specified"] = [-1e6]

# source
source = initialize_array("input", "source", 1)
source["id"] = [1]
source["node"] = [0]
source["status"] = [1]
source["u_ref"] = [1.0]

# voltage sensor
voltage_sensor = initialize_array("input", "sym_voltage_sensor", 1)
voltage_sensor["id"] = 5
voltage_sensor["measured_object"] = 0
voltage_sensor["u_sigma"] = [100.0]
voltage_sensor["u_measured"] = [10.5e3]

# power sensor
power_sensor = initialize_array("input", "sym_power_sensor", 2)
power_sensor["id"] = [6, 7]
power_sensor["measured_object"] = [2, 4]
power_sensor["measured_terminal_type"] = [
    MeasuredTerminalType.branch_from, MeasuredTerminalType.load
]
power_sensor["p_measured"] = [1e6, -1e6]
power_sensor["q_measured"] = [1e6, -1e6]
power_sensor["p_sigma"] = [1e4, 1e9]  # trust P for sensor 6
power_sensor["q_sigma"] = [1e9, 1e4]  # trust Q for sensor 7

# all
input_data = {
    "node": node,
    "line": line,
    "sym_load": sym_load,
    "source": source,
    "sym_voltage_sensor": voltage_sensor,
    "sym_power_sensor": power_sensor,
}



# Validation input (bug)

In [3]:
assert_valid_input_data(input_data=input_data, calculation_type=CalculationType.state_estimation)

ValidationException: There is a validation error in input_data:
	Field 'power_sigma' is missing for 2 sym_power_sensors.

# Build model

In [4]:
model = PowerGridModel(input_data)

# Iterative linear state estimation

We first try to run state estimation with iterative linear method. 
As expected, the algorithm does not differentiate accuracy for active and reactive power.
Therefore, it just averages both sensor value. The result is then zero for both P and Q.

In [5]:
ilse_result = model.calculate_state_estimation(calculation_method=CalculationMethod.iterative_linear)
print("-----Load----")
print(pd.DataFrame(ilse_result["sym_load"]))
print("-----Power Sensor----")
print(pd.DataFrame(ilse_result["sym_power_sensor"]))


-----Load----
   id  energized    p    q    i    s   pf
0   4          1 -0.0 -0.0  0.0  0.0  0.0
-----Power Sensor----
   id  energized  p_residual  q_residual
0   6          1   1000000.0   1000000.0
1   7          1  -1000000.0  -1000000.0


# Newton-Raphson state estimaiton

We then run the state estimation with Newton-Raphson method.
It can differentiate accuracy for active and reactive power.
So we trust the `power_sensor_6` for `p_measured` and `power_sensor_7` for `q_measured`.
In the end, the P is almost `1e6` and the Q is almost `-1e6` which is expected.

In [6]:
nrse_result = model.calculate_state_estimation(calculation_method=CalculationMethod.newton_raphson)
print("-----Load----")
print(pd.DataFrame(nrse_result["sym_load"]))
print("-----Power Sensor----")
print(pd.DataFrame(nrse_result["sym_power_sensor"]))

-----Load----
   id  energized              p              q          i             s  \
0   4          1  999981.865789 -999999.999799  77.747479  1.414201e+06   

       pf  
0  0.7071  
-----Power Sensor----
   id  energized    p_residual    q_residual
0   6          1  1.999427e-04  1.999637e+06
1   7          1 -1.999982e+06 -2.006559e-04
